# Notebook 06 — Minimal CZ Pulse Sequence Benchmark

This notebook adds a **piecewise pulse sequence** to the two-atom Rydberg model.

The goal is not to implement a full experimental-grade gate, but to move beyond constant driving and test a **more gate-like control protocol**.

We compare:
- a simple constant-drive evolution,
- a minimal three-step pulse sequence,

using the same entangling benchmark from Notebook 05.

A full neutral-atom CZ gate typically requires carefully engineered pulse timing, detuning, and phase control. This notebook provides a compact intermediate step.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qutip import basis, qeye, tensor, sigmax, sigmay, sesolve, mesolve

## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
sy = sigmay()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = {
    '|gg>': gg,
    '|gr>': gr,
    '|rg>': rg,
    '|rr>': rr,
}

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
sy1 = tensor(sy, I)
sy2 = tensor(I, sy)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

ideal_targets = {
    '|gg>': gg,
    '|gr>': gr,
    '|rg>': rg,
    '|rr>': -rr,
}


## Hamiltonian building blocks

In [ ]:
def two_atom_hamiltonian(omega1: float, omega2: float, delta: float, V: float, axis: str = 'x'):
    if axis == 'x':
        drive = 0.5 * omega1 * sx1 + 0.5 * omega2 * sx2
    elif axis == 'y':
        drive = 0.5 * omega1 * sy1 + 0.5 * omega2 * sy2
    else:
        raise ValueError("axis must be 'x' or 'y'")

    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction


def collapse_operators(gamma_decay: float = 0.0, gamma_phi: float = 0.0):
    c_ops = []
    if gamma_decay > 0:
        c_ops.append(np.sqrt(gamma_decay) * sm1)
        c_ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        c_ops.append(np.sqrt(gamma_phi) * n1)
        c_ops.append(np.sqrt(gamma_phi) * n2)
    return c_ops


## Piecewise pulse sequence

We define a three-step sequence:

1. pulse atom 1,
2. pulse atom 2 while interaction is present,
3. pulse atom 1 again.

This is only a toy control structure, but it is closer to a gate protocol than constant simultaneous driving.


In [ ]:
def run_segment(state, omega1: float, omega2: float, delta: float, V: float,
                duration: float, axis: str = 'x',
                gamma_decay: float = 0.0, gamma_phi: float = 0.0,
                n_steps: int = 120):
    H = two_atom_hamiltonian(omega1=omega1, omega2=omega2, delta=delta, V=V, axis=axis)
    c_ops = collapse_operators(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    times = np.linspace(0.0, duration, n_steps)

    if len(c_ops) == 0 and state.isket:
        result = sesolve(H, state, times)
    else:
        result = mesolve(H, state, times, c_ops)

    return result.states[-1], times, result.states


def run_minimal_sequence(psi0, omega: float, delta: float, V: float,
                         t_pi: float, t_mid: float,
                         axis: str = 'x',
                         gamma_decay: float = 0.0,
                         gamma_phi: float = 0.0):
    # Segment 1: pulse atom 1
    state1, times1, states1 = run_segment(
        psi0, omega1=omega, omega2=0.0, delta=delta, V=V,
        duration=t_pi, axis=axis,
        gamma_decay=gamma_decay, gamma_phi=gamma_phi
    )

    # Segment 2: pulse atom 2 with interaction active
    state2, times2, states2 = run_segment(
        state1, omega1=0.0, omega2=omega, delta=delta, V=V,
        duration=t_mid, axis=axis,
        gamma_decay=gamma_decay, gamma_phi=gamma_phi
    )

    # Segment 3: pulse atom 1 again
    state3, times3, states3 = run_segment(
        state2, omega1=omega, omega2=0.0, delta=delta, V=V,
        duration=t_pi, axis=axis,
        gamma_decay=gamma_decay, gamma_phi=gamma_phi
    )

    # Build concatenated timeline
    full_times = np.concatenate([
        times1,
        times2 + times1[-1],
        times3 + times1[-1] + times2[-1],
    ])
    full_states = states1 + states2 + states3
    return full_times, full_states, state3


## Benchmark helpers

In [ ]:
def state_overlap_score(final_state, target_state):
    if final_state.isket:
        amp = target_state.overlap(final_state)
        return float(np.abs(amp) ** 2)
    else:
        val = (target_state.dag() * final_state * target_state)
        if hasattr(val, 'full'):
            return float(np.real(val.full()[0, 0]))
        return float(np.real(val))


def benchmark_constant_drive(omega: float, delta: float, V: float,
                             t_gate: float,
                             gamma_decay: float = 0.0,
                             gamma_phi: float = 0.0,
                             n_steps: int = 240):
    scores = {}
    for label, psi0 in basis_states.items():
        H = two_atom_hamiltonian(omega1=omega, omega2=omega, delta=delta, V=V, axis='x')
        c_ops = collapse_operators(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
        times = np.linspace(0.0, t_gate, n_steps)

        if len(c_ops) == 0:
            result = sesolve(H, psi0, times)
        else:
            result = mesolve(H, psi0, times, c_ops)

        final_state = result.states[-1]
        scores[label] = state_overlap_score(final_state, ideal_targets[label])

    avg_score = float(np.mean(list(scores.values())))
    worst_score = float(np.min(list(scores.values())))
    return avg_score, worst_score, scores


def benchmark_pulse_sequence(omega: float, delta: float, V: float,
                             t_pi: float, t_mid: float,
                             gamma_decay: float = 0.0,
                             gamma_phi: float = 0.0):
    scores = {}
    for label, psi0 in basis_states.items():
        _, _, final_state = run_minimal_sequence(
            psi0,
            omega=omega,
            delta=delta,
            V=V,
            t_pi=t_pi,
            t_mid=t_mid,
            axis='x',
            gamma_decay=gamma_decay,
            gamma_phi=gamma_phi,
        )
        scores[label] = state_overlap_score(final_state, ideal_targets[label])

    avg_score = float(np.mean(list(scores.values())))
    worst_score = float(np.min(list(scores.values())))
    return avg_score, worst_score, scores


## Compare constant drive vs pulse sequence at one operating point

In [ ]:
omega = 1.0
delta = 0.0
V = 4.0
t_pi = np.pi / omega
t_mid = 2 * np.pi / omega
t_gate_const = 2 * t_pi + t_mid

avg_const, worst_const, scores_const = benchmark_constant_drive(
    omega=omega, delta=delta, V=V, t_gate=t_gate_const
)
avg_seq, worst_seq, scores_seq = benchmark_pulse_sequence(
    omega=omega, delta=delta, V=V, t_pi=t_pi, t_mid=t_mid
)

print('Constant drive:')
print(f'  average score = {avg_const:.4f}')
print(f'  worst-case score = {worst_const:.4f}')
print('Pulse sequence:')
print(f'  average score = {avg_seq:.4f}')
print(f'  worst-case score = {worst_seq:.4f}')


## Basis-state comparison

In [ ]:
labels = list(scores_const.keys())
x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8, 4.8))
plt.bar(x - width/2, [scores_const[k] for k in labels], width=width, label='constant drive')
plt.bar(x + width/2, [scores_seq[k] for k in labels], width=width, label='pulse sequence')
plt.xticks(x, labels)
plt.ylim(0, 1.05)
plt.ylabel('State-overlap score')
plt.title('Basis-state comparison: constant drive vs pulse sequence')
plt.legend()
plt.tight_layout()
plt.show()

## Scan the middle pulse duration

In [ ]:
t_mid_vals = np.linspace(0.2 * np.pi / omega, 3.5 * np.pi / omega, 40)
avg_scores = []
worst_scores = []

for t_mid_test in t_mid_vals:
    avg_score, worst_score, _ = benchmark_pulse_sequence(
        omega=omega,
        delta=delta,
        V=V,
        t_pi=t_pi,
        t_mid=t_mid_test,
    )
    avg_scores.append(avg_score)
    worst_scores.append(worst_score)

In [ ]:
plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, avg_scores, label='average score')
plt.plot(t_mid_vals, worst_scores, label='worst-case score')
plt.xlabel(r'Middle pulse duration $t_{mid}$')
plt.ylabel('Benchmark score')
plt.title('Dependence on the middle pulse duration')
plt.legend()
plt.tight_layout()
plt.show()

## Pulse-sequence landscape over $(\Omega, V)$

In [ ]:
omega_vals = np.linspace(0.4, 2.0, 24)
V_vals = np.linspace(0.0, 8.0, 24)

avg_landscape = np.zeros((len(V_vals), len(omega_vals)))
worst_landscape = np.zeros((len(V_vals), len(omega_vals)))

for i, V_scan in enumerate(V_vals):
    for j, omega_scan in enumerate(omega_vals):
        t_pi_scan = np.pi / omega_scan
        t_mid_scan = 2 * np.pi / omega_scan
        avg_score, worst_score, _ = benchmark_pulse_sequence(
            omega=omega_scan,
            delta=0.0,
            V=V_scan,
            t_pi=t_pi_scan,
            t_mid=t_mid_scan,
        )
        avg_landscape[i, j] = avg_score
        worst_landscape[i, j] = worst_score

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    avg_landscape,
    origin='lower',
    aspect='auto',
    extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]],
)
plt.contour(omega_vals, V_vals, avg_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Pulse-sequence average benchmark over $(\Omega, V)$')
plt.colorbar(im, label='average score')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    worst_landscape,
    origin='lower',
    aspect='auto',
    extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]],
)
plt.contour(omega_vals, V_vals, worst_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Pulse-sequence worst-case score over $(\Omega, V)$')
plt.colorbar(im, label='worst-case score')
plt.tight_layout()
plt.show()

## Noise sensitivity of the pulse sequence

In [ ]:
gamma_decay_grid = np.linspace(0.0, 0.12, 20)
gamma_phi_grid = np.linspace(0.0, 0.12, 20)

noise_avg = np.zeros((len(gamma_phi_grid), len(gamma_decay_grid)))

for i, gamma_phi in enumerate(gamma_phi_grid):
    for j, gamma_decay in enumerate(gamma_decay_grid):
        avg_score, _, _ = benchmark_pulse_sequence(
            omega=1.0,
            delta=0.0,
            V=4.0,
            t_pi=np.pi,
            t_mid=2*np.pi,
            gamma_decay=gamma_decay,
            gamma_phi=gamma_phi,
        )
        noise_avg[i, j] = avg_score

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    noise_avg,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_grid[0], gamma_decay_grid[-1], gamma_phi_grid[0], gamma_phi_grid[-1]],
)
plt.contour(gamma_decay_grid, gamma_phi_grid, noise_avg, colors='white', linewidths=0.4)
plt.xlabel(r'Spontaneous emission $\gamma$')
plt.ylabel(r'Pure dephasing $\gamma_\phi$')
plt.title(r'Pulse-sequence average benchmark over $(\gamma, \gamma_\phi)$')
plt.colorbar(im, label='average score')
plt.tight_layout()
plt.show()

## Interpretation

This notebook adds a more structured control protocol than constant driving.

Useful readouts are:
- whether the pulse sequence improves the worst-case basis-state score,
- whether tuning the middle pulse duration helps isolate more gate-like behavior,
- whether the pulse-sequence landscape is cleaner than the constant-drive benchmark.

Even if this simple sequence does not produce a high-fidelity CZ gate, it demonstrates the next conceptual step: **moving from static Hamiltonian evolution toward gate protocol design**.


## Optional next steps

Natural extensions are:

- add detuning ramps during the middle pulse,
- add phase shifts by switching between $\sigma_x$ and $\sigma_y$ control,
- replace this hand-built sequence with pulse optimization,
- compute a full unitary or process-fidelity benchmark.
